# Fresh Approach 3 experiments
Public pretrained initialization only. Development: 323 train-derived candidates. Final: 333 only after sealing. Read README first.


## 1. Setup
Choose the isolated school CUDA kernel. No model downloads or execution until RUN_DEVELOP=True.


In [ ]:
import json, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from approach3_experiment_suite.suite import Suite, final_evaluate
config = json.loads((ROOT / 'approach3_experiment_suite/config.json').read_text())
config['selected'] = ['R1']  # available: R1 F2 R5 D3 R4 R3 F3 F4
MODE = 'smoke'  # set 'full' only after small real-weight smoke succeeds
ALLOW_DOWNLOAD = False  # True explicitly permits public pretrained downloads
RUN_DEVELOP = False
RUN_FINAL = False


## 2. Data audit
Reads only train.csv. No official test labels or candidates are loaded. Original 7223/777 membership is preserved; reduced training/proxy split is recorded.


In [ ]:
suite = Suite(config, mode=MODE, allow_download=ALLOW_DOWNLOAD)
display(suite.catalog())
print('Run folder:', suite.path)
print('Development candidates:', len(suite.dev['pairs']))
print('Reduced train / proxy validation:', len(suite.dev['reduced']), len(suite.dev['holdout']))


## 3. Model rules
R1/F2/R5 and D3 use public frozen models. R4/R3/F3/F4 initialize fresh from public pretrained weights. No E1 or previous checkpoints are accepted. Every method must finish development before the suite is sealed.


In [ ]:
assert len(suite.dev['pairs']) == 323
assert not set(suite.dev['reduced'].pair_row) & set(suite.dev['holdout'].pair_row)


## 4. Development / training
Set RUN_DEVELOP=True in Setup, then rerun from Setup. Do not execute final until all decisions are frozen. Smoke results are mechanics only.


In [ ]:
if RUN_DEVELOP:
    development_path = suite.run_selected()
    print('Sealed development:', development_path)
else:
    print('Audit only; no weights loaded and no training started.')


## 5. Separate final evaluation
Use only a fully sealed directory. A full directory opens official tests and expands to 333 candidates. A smoke directory evaluates only train-derived proxies, without test access. Never select settings from this table.


In [ ]:
DEVELOPMENT_PATH = suite.path  # or Path('exact/path/to/a/sealed/run')
if RUN_FINAL:
    evaluation_path = final_evaluate(DEVELOPMENT_PATH, allow_download=ALLOW_DOWNLOAD)
    print('Evaluation:', evaluation_path)


## 6. Save / inspect
Keep the entire run folder, including newly trained checkpoints and the seal. Copy to persistent storage before leaving the school computer.


In [ ]:
if RUN_FINAL:
    import pandas as pd
    display(pd.read_csv(evaluation_path / 'all_comparisons.csv'))
else:
    print('Save this complete folder:', suite.path)
